# Классификация ботов — quickstart

Ноутбук показывает, как загрузить данные, собрать пару простейших признаков и получить
валидный `submission.csv`. Это **не** решение задачи: скор такого baseline будет чуть выше
константы. Дальше — ваша работа.

Условие и описание метрики — в `README.md`.

In [24]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score

from metric import precision_at_recall

SEED = 42

In [25]:
import numpy as np
import pandas as pd

train = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])

print(train.shape, test.shape, events.shape)
print('доля ботов в train:', train.target.mean().round(4))
train.head()

(11091, 5) (4909, 4) (328905, 14)
доля ботов в train: 0.0811


,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0


In [26]:
events.head()

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
0,ck_5efbea1befdefe1b,2026-04-26 09:11:24,200,item_view,desktop,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1027450.0,elektronika,kaliningrad,pro,NaN,NaN,NaN,NaN
1,ck_c4ca1434f3778f1d,2026-04-20 14:04:31,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,NaN,telefony,habarovsk,NaN,iphone 13 128,4.0,NaN,NaN
2,ck_d274382b19488771,2026-04-13 12:02:56,200,item_view,WEB,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1048743.0,kvartiry_prodazha,kirov,private,NaN,NaN,675.0,276.0
3,ck_fbbed2ff14944ce9,2026-04-08 06:12:14,100,search_results_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,bytovaya_tehnika,novosibirsk,NaN,пылесос dyson,2.0,NaN,NaN
4,ck_56cc15c7c634cb9f,2026-04-12 17:16:05,100,search_results_view,WEB,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,odezhda,sankt-peterburg,NaN,костюм мужской,2.0,NaN,NaN


## Осмотреться

Прежде чем считать агрегаты, стоит посмотреть на данные: какие типы событий бывают, что
лежит в `platform` и `user_agent`, где пропуски, всё ли уникально.

In [27]:
print(events.event_name.value_counts(), '\n')
print(events.platform.value_counts(), '\n')
print('пропуски по колонкам:')
print(events.isna().mean().round(3))

event_name
item_view               120817
search_results_view     100402
photo_swipe              36517
favorite_add             19049
seller_page_view         17403
contact_phone_show       11316
captcha_shown             7928
login                     6267
contact_chat_open         6125
contact_message_sent      3081
Name: count, dtype: int64 

platform
desktop    46866
WEB        46476
Web        46365
web        45951
ANDROID    42462
Android    42254
android    42159
iphone      4103
IOS         4100
iOS         4091
ios         4078
Name: count, dtype: int64 

пропуски по колонкам:
cookie_id        0.000
event_ts         0.000
eid              0.000
event_name       0.000
platform         0.000
user_agent       0.000
item_id          0.348
item_category    0.100
item_location    0.072
seller_type      0.407
search_query     0.695
search_page      0.695
pointer_x        0.670
pointer_y        0.670
dtype: float64


## Окно наблюдения

Признаки считаем только по событиям внутри окна: `window_start_ts <= event_ts < window_end_ts`.
Это требование из условия, а не рекомендация.

In [28]:
def events_in_window(events, meta):
    """Оставляем только события внутри окна наблюдения."""
    ev = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    return ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]

ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)
print(f"Событий в окне train: {len(ev_tr)}, test: {len(ev_te)}")

Событий в окне train: 198436, test: 89690


In [29]:
ev_tr.head()

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y,window_start_ts,window_end_ts
0,ck_d274382b19488771,2026-04-13 12:02:56,200,item_view,WEB,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1048743.0,kvartiry_prodazha,kirov,private,NaN,NaN,675.0,276.0,2026-04-13,2026-04-14
1,ck_fbbed2ff14944ce9,2026-04-08 06:12:14,100,search_results_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,bytovaya_tehnika,novosibirsk,NaN,пылесос dyson,2.0,NaN,NaN,2026-04-08,2026-04-09
2,ck_56cc15c7c634cb9f,2026-04-12 17:16:05,100,search_results_view,WEB,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,odezhda,sankt-peterburg,NaN,костюм мужской,2.0,NaN,NaN,2026-04-12,2026-04-13
4,ck_f80d85f038cdacee,2026-04-11 16:08:12,200,item_view,desktop,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1000845.0,bytovaya_tehnika,sankt-peterburg,private,NaN,NaN,NaN,NaN,2026-04-11,2026-04-12
5,ck_e4f046f721d10f81,2026-04-10 16:50:05,200,item_view,IOS,Mozilla/5.0 (iPhone; CPU iPhone OS 15_2 like M...,1055188.0,telefony,ekaterinburg,pro,NaN,NaN,NaN,NaN,2026-04-10,2026-04-11


## Простейшие признаки

Два агрегата — сколько событий и сколько разных объявлений. Этого заведомо мало.

In [30]:
def extract_features(ev, meta):
    """Извлекаем поведенческие, временные и технические признаки."""
    ev = ev.copy()
    
    # Сортируем для корректного расчёта time_diff
    ev = ev.sort_values(['cookie_id', 'event_ts'])
    
    # Временные признаки
    ev['time_diff'] = ev.groupby('cookie_id')['event_ts'].diff().dt.total_seconds()
    ev['instant_action'] = (ev['time_diff'] == 0).astype(int)
    ev['hour'] = ev['event_ts'].dt.hour
    ev['is_night'] = ((ev['hour'] >= 0) & (ev['hour'] < 6)).astype(int)
    
    # User-Agent признаки
    # Подозрительные UA (боты, headless-браузеры, скрипты)
    bot_ua_pattern = r'(?i)(bot|spider|crawler|headless|phantom|selenium|curl|wget|python|scrapy|java/|go-http|okhttp|libwww|httpunit)'
    ev['is_bot_ua'] = ev['user_agent'].str.contains(bot_ua_pattern, na=False).astype(int)
    ev['ua_length'] = ev['user_agent'].str.len().fillna(0)
    
    # Признаки курсора (pointer)
    ev['pointer_null'] = (ev['pointer_x'].isna() | ev['pointer_y'].isna()).astype(int)
    ev['pointer_zero'] = ((ev['pointer_x'] == 0) & (ev['pointer_y'] == 0)).astype(int)
    
    # Поведенческие признаки
    # Повторяющиеся поисковые запросы (боты часто ищут одно и то же)
    ev['query_repeat'] = ev.groupby(['cookie_id', 'search_query'])['event_ts'].transform('count')
    
    # Агрегация по cookie_id
    g = ev.groupby('cookie_id')
    
    f = pd.DataFrame({
        # Базовые счётчики
        'n_events': g.size(),
        'n_event_types': g['eid'].nunique(),
        'n_unique_agents': g['user_agent'].nunique(),
        
        # Разнообразие просмотров (важный поведенческий признак)
        'item_nunique': g['item_id'].nunique(),
        'category_nunique': g['item_category'].nunique(),
        'seller_nunique': g['seller_type'].nunique(),
        'query_nunique': g['search_query'].nunique(),
        'search_page_nunique': g['search_page'].nunique(),
        
        # Временные паттерны
        'time_diff_mean': g['time_diff'].mean(),
        'time_diff_std': g['time_diff'].std(),
        'time_diff_min': g['time_diff'].min(),
        'time_diff_median': g['time_diff'].median(),
        'instant_action_sum': g['instant_action'].sum(),
        
        # User-Agent
        'is_bot_ua_max': g['is_bot_ua'].max(),
        'is_bot_ua_mean': g['is_bot_ua'].mean(),
        'ua_length_mean': g['ua_length'].mean(),
        'ua_length_std': g['ua_length'].std(),
        
        # Курсор
        'pointer_null_mean': g['pointer_null'].mean(),
        'pointer_zero_mean': g['pointer_zero'].mean(),
        'pointer_x_std': g['pointer_x'].std(),
        'pointer_y_std': g['pointer_y'].std(),
        
        # Ночная активность
        'is_night_mean': g['is_night'].mean(),
        
        # Повторяемость запросов
        'query_repeat_max': g['query_repeat'].max(),
        'query_repeat_mean': g['query_repeat'].mean(),
        
        # Платформа
        'platform_nunique': g['platform'].nunique(),
    })
    
    # Слияние с метаданными (чтобы не потерять cookie без событий)
    meta = meta[['cookie_id', 'cookie_created_at', 'window_start_ts', 'window_end_ts']].copy()
    meta['window_duration_sec'] = (meta['window_end_ts'] - meta['window_start_ts']).dt.total_seconds()
    meta['cookie_age_sec'] = (meta['window_start_ts'] - meta['cookie_created_at']).dt.total_seconds()
    
    f = meta.merge(f.reset_index(), on='cookie_id', how='left')
    
    # Производные признаки
    f['events_per_minute'] = f['n_events'] / (f['window_duration_sec'] / 60 + 1e-6)
    f['instant_action_ratio'] = f['instant_action_sum'] / (f['n_events'] + 1e-6)
    f['item_diversity'] = f['item_nunique'] / (f['n_events'] + 1e-6)
    f['query_diversity'] = f['query_nunique'] / (f['n_events'] + 1e-6)
    f['unique_agents_ratio'] = f['n_unique_agents'] / (f['n_events'] + 1e-6)
    
    # Заполняем пропуски нулями (для cookie без событий)
    f = f.fillna(0)
    
    return f

print("Генерация признаков")
Xtr = extract_features(ev_tr, train)
Xte = extract_features(ev_te, test)
ytr = train.target.values

# Колонки для обучения (только числовые, исключаем служебные)
meta_cols = ['cookie_id', 'window_start_ts', 'window_end_ts']
feature_cols = [c for c in Xtr.columns if c not in meta_cols and Xtr[c].dtype in ['int64', 'float64', 'int32', 'float32']]

print(f"Признаков: {len(feature_cols)}")

Генерация признаков
Признаков: 32


## Валидация

Тест лежит **позже** трейна по времени, поэтому и валидацию честно делать по времени, а не
случайным сплитом.

Метрику берём из `metric.py` — это ровно тот код, которым считает проверяющая система.
Своя реализация почти наверняка разойдётся с официальной на одинаковых `score`:
их нельзя разделять, группа равных значений отмечается целиком.

In [31]:
from sklearn.ensemble import RandomForestClassifier
from metric import precision_at_recall   

is_valid = train.window_start_ts.ge('2026-04-17').values
cols = feature_cols 

model = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=0)
model.fit(Xtr.loc[~is_valid, cols], ytr[~is_valid])
p_va = model.predict_proba(Xtr.loc[is_valid, cols])[:, 1]

print('P@R0.7 на валидации:', round(precision_at_recall(ytr[is_valid], p_va), 4))
print('доля ботов (это уровень константы):', round(ytr[is_valid].mean(), 4))

P@R0.7 на валидации: 0.7273
доля ботов (это уровень константы): 0.082


## Сабмит

In [32]:
model.fit(Xtr[cols], ytr)
sub = pd.DataFrame({
    'cookie_id': Xte.cookie_id,
    'score': model.predict_proba(Xte[cols])[:, 1],
})
assert len(sub) == len(test) and sub.score.between(0, 1).all()
sub.to_csv('submission.csv', index=False)
sub.head()

,cookie_id,score
0,ck_315fb710a0e371e7,0.001778
1,ck_a76ee3b3e3e522fd,0.066879
2,ck_94c9a4d382689e82,0.029836
3,ck_8eaf9509ad9462a0,0.006652
4,ck_9a88a5a989cb5bc6,0.047137


## Куда копать дальше

Подсказок по конкретным признакам не будет — это и есть содержание задания. Несколько
вопросов, которые стоит себе задать:

* чем поток событий робота отличается от потока событий человека, если смотреть не на
  количество, а на **моменты времени**;
* что полезного лежит в строке `user_agent` и почему её нельзя брать как есть;
* насколько разнообразно то, что смотрит кука: объявления, категории, запросы, страницы выдачи;
* всё ли в порядке с самим файлом событий — порядок строк, дубликаты, пропуски;
* какие признаки бесполезны, потому что описывают технические характеристики, а не поведение.

In [34]:
scale_pos_weight = (ytr[~is_valid] == 0).sum() / max((ytr[~is_valid] == 1).sum(), 1)

In [35]:
print("Обучение модели")
model = lgb.LGBMClassifier(
    n_estimators=1500,
    learning_rate=0.03,
    max_depth=7,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=20,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    random_state=SEED,
    n_jobs=-1,
    metric='auc',
    verbose=-1
)

# Обучаем на всём трейне (для финального сабмита)
model.fit(Xtr[feature_cols], ytr)

# Предсказания на валидации (для оценки)
model_val = lgb.LGBMClassifier(
    n_estimators=1500,
    learning_rate=0.03,
    max_depth=7,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=20,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    random_state=SEED,
    n_jobs=-1,
    metric='auc',
    verbose=-1
)
model_val.fit(Xtr.loc[~is_valid, feature_cols], ytr[~is_valid])
p_va = model_val.predict_proba(Xtr.loc[is_valid, feature_cols])[:, 1]

# Оценка по официальной метрике
p_at_r = precision_at_recall(ytr[is_valid], p_va)
auc_val = roc_auc_score(ytr[is_valid], p_va)
print(f"P@R0.7 на валидации: {p_at_r:.4f}")
print(f"ROC-AUC на валидации: {auc_val:.4f}")
print(f"Доля ботов (уровень константы): {ytr[is_valid].mean():.4f}")

Обучение модели...
P@R0.7 на валидации: 0.6570
ROC-AUC на валидации: 0.9074
Доля ботов (уровень константы): 0.0820


## Сабмит


In [36]:
# Калибровка важна для P@R: предсказания должны быть близки к реальным вероятностям
print("Калибровка вероятностей...")
calibrator = CalibratedClassifierCV(model, method='isotonic', cv=3)
calibrator.fit(Xtr[feature_cols], ytr)

Калибровка вероятностей...


,"estimator estimator: estimator instance, default=NoneThe classifier whose output need to be calibrated to provide moreaccurate `predict_proba` outputs. The default classifier isa :class:`~sklearn.svm.LinearSVC`... versionadded:: 1.2",LGBMClassifie... verbose=-1)
,"method method: {'sigmoid', 'isotonic', 'temperature'}, default='sigmoid'The method to use for calibration. Can be:- 'sigmoid', which corresponds to Platt's method (i.e. a binary logistic regression model).- 'isotonic', which is a non-parametric approach.- 'temperature', temperature scaling.Sigmoid and isotonic calibration methods natively support only binaryclassifiers and extend to multi-class classification using a One-vs-Rest (OvR)strategy with post-hoc renormalization, i.e., adjusting the probabilities aftercalibration to ensure they sum up to 1.In contrast, temperature scaling naturally supports multi-class calibration byapplying `softmax(classifier_logits/T)` with a value of `T` (temperature)that optimizes the log loss.For very uncalibrated classifiers on very imbalanced datasets, sigmoidcalibration might be preferred because it fits an additional interceptparameter. This helps shift decision boundaries appropriately when theclassifier being calibrated is biased towards the majority class.Isotonic calibration is not recommended when the number of calibration samplesis too low ``(≪1000)`` since it then tends to overfit... versionchanged:: 1.8 Added option 'temperature'.",'isotonic'
,"cv cv: int, cross-validation generator, or iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- integer, to specify the number of folds,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used. If ``y`` isneither binary nor multiclass, :class:`~sklearn.model_selection.KFold`is used.Refer to the :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors.Base estimator clones are fitted in parallel across cross-validationiterations.See :term:`Glossary <n_jobs>` for more details... versionadded:: 0.24",None
,"ensemble ensemble: bool, or ""auto"", default=""auto""Determines how the calibrator is fitted.""auto"" will use `False` if the `estimator` is a:class:`~sklearn.frozen.FrozenEstimator`, and `True` otherwise.If `True`, the `estimator` is fitted using training data, andcalibrated using testing data, for each `cv` fold. The final estimatoris an ensemble of `n_cv` fitted classifier and calibrator pairs, where`n_cv` is the number of cross-validation folds. The output is theaverage predicted probabilities of all pairs.If `False`, `cv` is used to compute unbiased predictions, via:func:`~sklearn.model_selection.cross_val_predict`, which are thenused for calibration. At prediction time, the classifier used is the`estimator` trained on all the data.Note that this method is also internally implemented in:mod:`sklearn.svm` estimators with the `probabilities=True` parameter... versionadded:: 0.24.. versionchanged:: 1.6 `""auto""` option is added and is the default.",'auto'
Name,Type,Value
"calibrated_classifiers_ calibrated_classifiers_: list (len() equal to cv or 1 if `ensemble=False`)The list of classifier and calibrator pairs.- When `ensemble=True`, `n_cv` fitted `estimator` and calibrator pairs. `n_cv` is the number of cross-validation folds.- When `ensemble=False`, the `estimator`, fitted on all the data, and fitted calibrator... versionchanged:: 0.24 Single calibrated classifier case when `ensemble=False`.",list,"[<sklearn.cali...001E7EBDE6BA0>, <skl

In [37]:
print("Генерация submission.csv...")
scores = calibrator.predict_proba(Xte[feature_cols])[:, 1]

sub = pd.DataFrame({
    'cookie_id': Xte['cookie_id'],
    'score': scores,
})

# Проверки из условия
assert len(sub) == len(test), "Неверное количество строк в submission"
assert sub['cookie_id'].nunique() == len(sub), "Есть дубликаты cookie_id"
assert sub['score'].between(0, 1).all(), "Score вне диапазона [0, 1]"
assert not sub['score'].isna().any(), "Есть пропуски в score"

sub.to_csv('submission.csv', index=False)
print(f"Submission сохранён. Строк: {len(sub)}, score range: [{scores.min():.4f}, {scores.max():.4f}]")


Генерация submission.csv...
Submission сохранён. Строк: 4909, score range: [0.0000, 1.0000]
